# Complete End-to-End Erosion Prediction Pipeline

**Full workflow from data collection to erosion predictions**

## Pipeline Steps:

### 1️⃣ Data Collection (WFSDataBundler)
- Fetch WFS data for all scope regions
- Aggregate and deduplicate geometries
- Save to GeoPackage

### 2️⃣ Feature Engineering (DataHandler)
- Load WFS data from saved GeoPackage
- Generate WFS features for each region
- Load erosion data (river bank measurements)
- Process erosion features

### 3️⃣ Baseline Model Training & Prediction
- Train baseline erosion model
- Predict future erosion (10 time steps)
- Analyze predictions and identify critical regions

---

⚠️ **Important:** This notebook processes ALL regions in full scope.
For testing, use `test_wfs_bundler.ipynb` or `test_datahandler_geopackage_loading.ipynb` instead.

## Setup

In [4]:
# Additional imports for visualization and modeling
import src.model.baseline_model as BM
import folium
from folium.plugins import TimestampedGeoJson
import altair as alt
import shapely
from shapely.geometry import Polygon, LineString, Point

print("✅ Visualization libraries loaded!")

✅ Visualization libraries loaded!


In [2]:
import sys
sys.path.append("../../")

import src.paths as PATHS
import src.constants as CONST
import src.data.config as DATA_CONFIG
import src.data.wfs_bundler as WFS_BUNDLER
import src.data.data_handler as DH
import src.utils as U

import geopandas as gpd
import pandas as pd
from pathlib import Path
from datetime import datetime
import time

print("✅ Imports complete!")

/Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports complete!


## Configuration

In [3]:
# Input GeoPackage (contains scope regions and river centerline)
INPUT_GPKG = "phase1_2025-08-14_v1.gpkg"  # Change this to your input file
input_path = PATHS.DATA_DIR / INPUT_GPKG

# Output GeoPackage (will contain WFS data + features)
date_suffix = datetime.now().strftime("%Y%m%d")
output_filename = INPUT_GPKG.replace(".gpkg", f"_complete_{date_suffix}.gpkg")
output_path = PATHS.DATA_DIR / output_filename

print(f"📂 Input:  {input_path}")
print(f"📂 Output: {output_path}")
print(f"\n⚠️  Output file will be created (or overwritten if it exists)")

📂 Input:  /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/phase1_2025-08-14_v1.gpkg
📂 Output: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/phase1_2025-08-14_v1_complete_20260213.gpkg

⚠️  Output file will be created (or overwritten if it exists)


## Load Scope Regions

In [56]:
# Load scope regions
scope_regions = gpd.read_file(input_path, layer="vlakken_scope")

# Determine ID column name (flexible for different datasets)
id_col = None
for possible_id in ['position_id', 'location_id', 'region_id', 'id']:
    if possible_id in scope_regions.columns:
        id_col = possible_id
        break

if id_col is None:
    raise ValueError(f"No ID column found. Available columns: {list(scope_regions.columns)}")

print(f"📍 Loaded {len(scope_regions)} scope regions")
print(f"📐 CRS: {scope_regions.crs}")
print(f"🔑 ID column: '{id_col}'")
print(f"\n🗺️  First 3 region IDs: {scope_regions[id_col].head(3).tolist()}")

scope_regions.head(3)

📍 Loaded 233 scope regions
📐 CRS: EPSG:28992
🔑 ID column: 'location_id'

🗺️  First 3 region IDs: ['maas_l_2180_2181', 'maas_l_2181_2182', 'maas_l_2182_2183']


,location_id,start_year,end_year,geometry
0,maas_l_2180_2181,2016,2024,"POLYGON ((148604.881 416309.328, 148538.207 41..."
1,maas_l_2181_2182,2016,2024,"POLYGON ((148491.678 416329.661, 148416.977 41..."
2,maas_l_2182_2183,2016,2024,"POLYGON ((148373.399 416364.687, 148290.662 41..."


## Initialize Configuration

In [57]:
# Initialize data configuration
config = DATA_CONFIG.DataConfiguration()

print(f"📋 Configuration loaded")
print(f"\n🌐 WFS Services ({len(config.known_wfs_services)}):")
for i, service in enumerate(config.known_wfs_services, 1):
    layers = ", ".join(service.relevant_layers)
    print(f"{i}. {service.name}")
    print(f"   URL: {service.url}")
    print(f"   Layers: {layers}")

📋 Configuration loaded

🌐 WFS Services (3):
1. land_use
   URL: https://service.pdok.nl/rvo/brpgewaspercelen/wfs/v1_0
   Layers: BrpGewas
2. building_location
   URL: https://service.pdok.nl/lv/bag/wfs/v2_0
   Layers: bag:pand
3. vegetation
   URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_vegetatielegger/ows?version=2.0.0
   Layers: rws_vegetatielegger:bomen, rws_vegetatielegger:heggen, rws_vegetatielegger:vegetatieklassen


---

# STEP 1: Data Collection with WFSDataBundler

This step fetches WFS data for all regions and saves to a GeoPackage.

⏱️ **Estimated time:** ~1.6 seconds per region
- 100 regions: ~3 minutes
- 500 regions: ~15 minutes
- 12,130 regions: ~5-6 hours

In [13]:
# Initialize WFSDataBundler
bundler = WFS_BUNDLER.WFSDataBundler(
    scope_regions=scope_regions,
    config=config
)

print(f"✅ WFSDataBundler initialized")
print(f"📊 Ready to process {len(scope_regions)} regions")

✅ WFSDataBundler initialized
📊 Ready to process 233 regions


In [15]:
# Fetch WFS data for ALL regions (full scope)
print("🚀 Starting WFS data collection...")
print(f"⏱️  Estimated time: ~{len(scope_regions) * 1.6 / 60:.1f} minutes\n")

start_time = time.time()

bundler.fetch_all_regions(
    test_mode=False
)

elapsed = time.time() - start_time
print(f"\n⏱️  Total collection time: {elapsed / 60:.1f} minutes ({elapsed / len(scope_regions):.2f}s per region)")

🚀 Starting WFS data collection...
⏱️  Estimated time: ~6.2 minutes



Fetching WFS data: 100%|██████████| 233/233 [06:19<00:00,  1.63s/it]


⏱️  Total collection time: 6.3 minutes (1.63s per region)


In [16]:
# Show summary statistics
bundler.get_summary_stats()

{'num_services': 3,
 'num_regions_processed': 233,
 'num_successful': 233,
 'num_failed': 0,
 'services': {'land_use': {'num_layers': 1,
   'layers': {'BrpGewas': {'num_regions': 226, 'total_features': 557}}},
  'building_location': {'num_layers': 1,
   'layers': {'bag:pand': {'num_regions': 26, 'total_features': 140}}},
  'vegetation': {'num_layers': 3,
   'layers': {'rws_vegetatielegger:bomen': {'num_regions': 107,
     'total_features': 334},
    'rws_vegetatielegger:heggen': {'num_regions': 7, 'total_features': 28},
    'rws_vegetatielegger:vegetatieklassen': {'num_regions': 233,
     'total_features': 1963}}}}}

In [17]:
# Save WFS data to GeoPackage
print(f"💾 Saving WFS data to: {output_path.name}\n")

# First, copy the original GeoPackage to preserve existing layers
import shutil
shutil.copy2(input_path, output_path)
print(f"✅ Copied original layers from {input_path.name}")

# Now add WFS layers
bundler.save_to_geopackage(output_path)

print(f"\n✅ Complete! WFS data saved to: {output_path}")

💾 Saving WFS data to: phase1_2025-08-14_v1_complete_20260127.gpkg

✅ Copied original layers from phase1_2025-08-14_v1.gpkg

✅ Complete! WFS data saved to: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/phase1_2025-08-14_v1_complete_20260127.gpkg


In [18]:
# List all layers in the output GeoPackage
bundler.list_geopackage_layers(output_path)

{'original': ['beschermde_oever',
  'building_location/bag:pand',
  'land_use/BrpGewas',
  'middenlijn',
  'punten_oever',
  'summary_layer',
  'vegetation/rws_vegetatielegger:bomen',
  'vegetation/rws_vegetatielegger:heggen',
  'vegetation/rws_vegetatielegger:vegetatieklassen',
  'vlakken_erosie',
  'vlakken_scope'],
 'wfs': [],
 'all': ['beschermde_oever',
  'building_location/bag:pand',
  'land_use/BrpGewas',
  'middenlijn',
  'punten_oever',
  'summary_layer',
  'vegetation/rws_vegetatielegger:bomen',
  'vegetation/rws_vegetatielegger:heggen',
  'vegetation/rws_vegetatielegger:vegetatieklassen',
  'vlakken_erosie',
  'vlakken_scope']}

---

# STEP 2: Feature Engineering with DataHandler

This step loads the saved WFS data and generates ML-ready features.

⚡ **Fast!** ~0.01 seconds per region (200x faster than live WFS fetching)

In [82]:
# Load prediction regions from the output GeoPackage
prediction_regions = gpd.read_file(output_path, layer="vlakken_scope")

# Load river centerline (needed for bend calculation and erosion projections)
# NOTE: Use the erosion-specific centerline from Levering_erosie_data.gpkg
etienne_geospatial_data = PATHS.DATA_DIR / "Levering_erosie_data.gpkg"
centerline_layer_name = "Centreline_River"
river_centerline = gpd.read_file(etienne_geospatial_data, layer=centerline_layer_name)

# Align CRS with prediction regions
river_centerline.to_crs(prediction_regions.crs, inplace=True)

local_geospatial_data = {
    CONST.AggregationOperations.CENTERLINE_SHAPE.value: river_centerline
}

print(f"📍 Loaded {len(prediction_regions)} prediction regions")
print(f"📐 CRS: {prediction_regions.crs}")
print(f"🗺️  River centerline: {len(river_centerline)} segments (from {etienne_geospatial_data.name})")

📍 Loaded 233 prediction regions
📐 CRS: EPSG:28992
🗺️  River centerline: 13 segments (from Levering_erosie_data.gpkg)


In [83]:
# Initialize DataHandler
data_handler = DH.DataHandler(
    prediction_regions=prediction_regions,
    local_data_for_enrichment=local_geospatial_data,
    config=config,
)

print("✅ DataHandler initialized")

Setting the number extra features to 0, even though it should be automatically calculated.


✅ DataHandler initialized


In [84]:
# Load WFS data from GeoPackage and generate features
print("🚀 Loading WFS data and generating features...\n")

start_time = time.time()

data_handler.load_remote_data_from_geopackage(
    gpkg_path=output_path,
    show_progress=True
)

elapsed = time.time() - start_time
print(f"\n⏱️  Feature generation time: {elapsed:.2f} seconds ({elapsed / len(prediction_regions):.4f}s per region)")
print(f"📊 Output shape: {data_handler.scope_region_features.shape}")

🚀 Loading WFS data and generating features...



Processing regions:   0%|          | 0/233 [00:00<?, ?region/s]The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
The geodataframe is empty and thus the density cannot be calculated. It is assumed it would be 0.
Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cannot be calculated. It is as


⏱️  Feature generation time: 1.47 seconds (0.0063s per region)
📊 Output shape: (233, 11)


In [85]:
# Show feature columns
print("\n📋 Generated Feature Columns:")
print("=" * 60)

feature_cols = [col for col in data_handler.scope_region_features.columns 
                if col not in ['location_id', 'position_id', 'region_id', 'id', 'geometry', 'start_year', 'end_year']]

for i, col in enumerate(feature_cols, 1):
    print(f"{i:2}. {col}")

print(f"\n✅ Total features: {len(feature_cols)}")


📋 Generated Feature Columns:
 1. BrpGewas_area_fraction
 2. BrpGewas_majority_class_category
 3. BrpGewas_majority_class_gewas
 4. bag:pand_area_fraction
 5. bag:pand_majority_class_gebruiksdoel
 6. rws_vegetatielegger:bomen_numerical_density
 7. rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse

✅ Total features: 7


---

# STEP 3: Add Erosion Data & Train Baseline Model

Now we'll add erosion data (river bank measurements) and train a baseline erosion prediction model.

In [86]:
# Load erosion data (river bank point measurements over time)
print("📍 Loading erosion data...")

# Load river bank points from the GeoPackage
river_bank_locations = gpd.read_file(output_path, layer="punten_oever")

# Load erosion border (the limit line beyond which erosion is critical)
erosion_border_gdf = gpd.read_file(PATHS.DATA_DIR / "erosion_border_20250129.gpkg", layer="Tekenen_signaallijn_20250129")
erosion_border_gdf.to_crs(prediction_regions.crs, inplace=True)
erosion_border = erosion_border_gdf.iloc[0]["geometry"]  # Use first linestring

print(f"✅ Loaded {len(river_bank_locations)} river bank points")
print(f"✅ Loaded erosion border: {erosion_border.geom_type}")

river_bank_locations.head(3)

📍 Loading erosion data...
✅ Loaded 70495 river bank points
✅ Loaded erosion border: LineString


,type_oever,status,location_id,dtm_version,dtm_date,geometry
0,MILD_SLOPE,OK,maas_l_2180_2181,AHN3,2016,POINT Z (148573.428 416375.97 2.512)
1,MEDIUM_SLOPE,OK,maas_l_2180_2181,AHN3,2016,POINT Z (148572.932 416376.041 2.521)
2,STEEP_SLOPE,OK,maas_l_2180_2181,AHN3,2016,POINT Z (148572.4 416375.865 2.618)


In [87]:
# Re-initialize DataHandler with erosion data for baseline model
# NOTE: We keep the WFS features we already generated, and now add erosion features

print("🔄 Re-initializing DataHandler with erosion data...")

# Create baseline configuration (simpler than full config)
baseline_config = DATA_CONFIG.DataConfiguration(
    no_of_points_for_distance_calculation=CONST.DEFAULT_NO_OF_POINTS_FOR_DISTANCE_CALCULATION,
    prediction_region_id_column_name=CONST.PREDICTION_REGION_ID,
    timestamp_column_name="dtm_date",  # Column name in river_bank_locations
    use_only_certain_river_bank_points=CONST.DEFAULT_USE_ONLY_CERTAIN_RIVER_BANK_POINTS,
)

# Create new DataHandler with ALL data (WFS + erosion)
data_handler_full = DH.DataHandler(
    config=baseline_config,
    prediction_regions=prediction_regions,
    local_data_for_enrichment=local_geospatial_data,
    erosion_data=river_bank_locations,
    erosion_border=erosion_border,
)

print(f"✅ DataHandler initialized with erosion data")

Setting the number extra features to 0, even though it should be automatically calculated.


🔄 Re-initializing DataHandler with erosion data...
✅ DataHandler initialized with erosion data


In [88]:
# Load WFS features we generated earlier
print("🚀 Loading WFS data from GeoPackage...")

start_time = time.time()

data_handler_full.load_remote_data_from_geopackage(
    gpkg_path=output_path,
    show_progress=True
)

elapsed = time.time() - start_time
print(f"\n⏱️  WFS loading time: {elapsed:.2f} seconds")
print(f"📊 WFS features shape: {data_handler_full.scope_region_features.shape}")

🚀 Loading WFS data from GeoPackage...


Processing regions:   0%|          | 0/233 [00:00<?, ?region/s]The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
The geodataframe is empty and thus the density cannot be calculated. It is assumed it would be 0.
Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cannot be calculated. It is as


⏱️  WFS loading time: 1.69 seconds
📊 WFS features shape: (233, 11)


In [89]:
# Process erosion features (calculate distances to erosion border over time)
print("🚀 Processing erosion features...")

data_handler_full.process_erosion_features()

print(f"✅ Erosion features processed")
print(f"📊 Erosion data shape: {data_handler_full.processed_erosion_data.shape}")

# Preview erosion features
data_handler_full.processed_erosion_data.head()

🚀 Processing erosion features...
✅ Erosion features processed
📊 Erosion data shape: (539, 2)


distance_to_erosion_border  \
location_id        dtm_date                               
ijssel_l_9509_9510 2017                   -90582.620209   
                   2022                   -90581.745671   
ijssel_l_9510_9511 2017                   -90608.044175   
                   2022                   -90607.025205   
ijssel_l_9520_9521 2017                   -90913.780483   

                             timesteps_since_last_measurement  
location_id        dtm_date                                    
ijssel_l_9509_9510 2017                                   1.0  
                   2022                                   5.0  
ijssel_l_9510_9511 2017                                   1.0  
                   2022                                   5.0  
ijssel_l_9520_9521 2017                                   1.0

## Train Baseline Erosion Model

In [66]:
# Import baseline model module
import src.model.baseline_model as BM

# Initialize and train baseline model
print("🚀 Training baseline erosion model...")

baseline_model = BM.BaselineErosionModel(
    config=baseline_config,
    training_data=data_handler_full.processed_erosion_data,
    verbose=True,
)

baseline_model.train()

print("\n✅ Model trained successfully!")

🚀 Training baseline erosion model...

✅ Model trained successfully!


In [67]:
baseline_model.model

{'ijssel_l_9509_9510': np.float64(-0.25171287497505546),
 'ijssel_l_9510_9511': np.float64(-0.07930835781735368),
 'ijssel_l_9520_9521': np.float64(0.4768482574232621),
 'ijssel_l_9526_9527': np.float64(-0.010189461358822881),
 'ijssel_l_9527_9528': np.float64(0.19303018575592432),
 'ijssel_l_9528_9529': np.float64(0.2748628927365644),
 'ijssel_r_9478_9479': np.float64(-1.6001980369153899),
 'ijssel_r_9479_9480': np.float64(-2.8870483743725344),
 'ijssel_r_9480_9481': np.float64(3.780520810422604),
 'ijssel_r_9482_9483': np.float64(-1.153563061830937),
 'ijssel_r_9487_9488': np.float64(-1.7458824364293832),
 'ijssel_r_9488_9489': np.float64(-2.7493620988796463),
 'ijssel_r_9500_9501': np.float64(0.039074687112588435),
 'ijssel_r_9501_9502': np.float64(-2.0683603187440895),
 'ijssel_r_9502_9503': np.float64(-2.0178250544704497),
 'ijssel_r_9503_9504': np.float64(-2.479467609146377),
 'ijssel_r_9504_9505': np.float64(-1.8439146746764892),
 'ijssel_r_9505_9506': np.float64(-1.903977440038

## Make Predictions

Predict future erosion for the next 10 time steps.

In [68]:
# Prepare data for prediction (use the most recent time step)
data_for_prediction = data_handler_full.processed_erosion_data.copy()

# Make predictions for the next 10 time steps
print("🔮 Making predictions for next 10 time steps...")

predictions = baseline_model.predict(data_for_prediction, prediction_length=10)

print(f"✅ Predictions complete!")
print(f"📊 Prediction shape: {predictions.shape}")

# Show predictions for first 5 regions
predictions.head()

🔮 Making predictions for next 10 time steps...
✅ Predictions complete!
📊 Prediction shape: (539, 10)


future_distance_to_erosion_border_1  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90575.142575   
                   2022                             90573.884011   
ijssel_l_9510_9511 2017                             90581.965323   
                   2022                             90581.568781   
ijssel_l_9520_9521 2017                             90902.740899   

                             future_distance_to_erosion_border_2  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90574.890862   
                   2022                             90573.632298   
ijssel_l_9510_9511 2017                             90581.886014   
                   2022                             90581.489472   
ijssel_l_9520_9521 2017                             90903.217747   

                             future_distance_to_erosion_border_3  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90574.639149   
                   2022                             90573.380585   
ijssel_l_9510_9511 2017                             90581.806706   
                   2022                             90581.410164   
ijssel_l_9520_9521 2017                             90903.694596   

                             future_distance_to_erosion_border_4  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90574.387437   
                   2022                             90573.128872   
ijssel_l_9510_9511 2017                             90581.727397   
                   2022                             90581.330856   
ijssel_l_9520_9521 2017                             90904.171444   

                             future_distance_to_erosion_border_5  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90574.135724   
                   2022                             90572.877159   
ijssel_l_9510_9511 2017                             90581.648089   
                   2022                             90581.251547   
ijssel_l_9520_9521 2017                             90904.648292   

                             future_distance_to_erosion_border_6  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90573.884011   
                   2022                             90572.625446   
ijssel_l_9510_9511 2017                             90581.568781   
                   2022                             90581.172239   
ijssel_l_9520_9521 2017                             90905.125140   

                             future_distance_to_erosion_border_7  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90573.632298   
                   2022                             90572.373734   
ijssel_l_9510_9511 2017                             90581.489472   
                   2022                             90581.092931   
ijssel_l_9520_9521 2017                             90905.601989   

                             future_distance_to_erosion_border_8  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90573.380585   
                   2022                             90572.122021   
ijssel_l_9510_9511 2017                             90581.410164   
                   2022                             90581.013622   
ijssel_l_9520_9521 2017                             90906.078837   

                             future_distance_to_erosion_border_9  \
location_id        dtm_date                                        
ijssel_l_9509_9510 2017                             90573.128872   
         

## Analyze Predictions

In [69]:
# Calculate prediction statistics
print("📊 Prediction Statistics")
print("=" * 60)

# Get the 10th time step predictions (furthest in future)
future_10 = predictions['future_distance_to_erosion_border_10']

print(f"\nDistance to erosion border (10 steps ahead):")
print(f"  Mean: {future_10.mean():.2f} meters")
print(f"  Median: {future_10.median():.2f} meters")
print(f"  Min: {future_10.min():.2f} meters")
print(f"  Max: {future_10.max():.2f} meters")

# Count critical regions (negative distance = crossed erosion border)
critical_regions = (future_10 < 0).sum()
print(f"\n⚠️  Critical regions (crossed border): {critical_regions} / {len(future_10)} ({100*critical_regions/len(future_10):.1f}%)")

# Show worst 5 regions
print(f"\n🚨 Top 5 regions at risk (most erosion):")
worst_regions = future_10.nsmallest(5)
for (location_id, ahn), distance in worst_regions.items():
    print(f"  {location_id}: {distance:.2f} m")

📊 Prediction Statistics

Distance to erosion border (10 steps ahead):
  Mean: -6332.35 meters
  Median: -2494.84 meters
  Min: -91611.69 meters
  Max: 91464.71 meters

⚠️  Critical regions (crossed border): 370 / 539 (68.6%)

🚨 Top 5 regions at risk (most erosion):
  nevengeul_ijssel_l_0014_0015: -91611.69 m
  nevengeul_ijssel_l_0014_0015: -91611.58 m
  nevengeul_ijssel_l_0011_0012: -91603.42 m
  nevengeul_ijssel_l_0011_0012: -91603.29 m
  ijssel_r_9527_9528: -91594.57 m


---

# Visualization

Create an interactive map showing predicted erosion over time.

In [70]:
# Helper functions for visualization

def project_scope_onto_erosion_border(scope_polygon: Polygon, erosion_border: LineString) -> LineString:
    """Get a section of the erosion border that corresponds to the scope polygon onto it.
    
    :param scope_polygon: polygon within which we predict the bank erosion
    :param erosion_border: the erosion border
    :returns: a section of the erosion border
    """
    scope_points = list(scope_polygon.exterior.coords)

    projected_distances = []
    for scope_point in scope_points:
        distance_along_erosion_border = erosion_border.project(Point(scope_point))
        projected_distances.append(distance_along_erosion_border)

    min_distance = min(projected_distances)
    max_distance = max(projected_distances)

    starting_point = erosion_border.interpolate(min_distance)
    ending_point = erosion_border.interpolate(max_distance)

    projection_line = LineString([starting_point, ending_point])

    return projection_line


def move_projected_scope_to_side(projected_scope: LineString, distance: float, river_centerline: LineString) -> LineString:
    """Using the distance from the erosion border, get where the projected scope polygon would lie as a surrogate to the river bank.
    
    :param projected_scope: line that projects the scope polygon onto the erosion border
    :param distance: distance between the (predicted) river bank and the erosion border
    :param river_centerline: the centerline to determine the direction of shifting
    """
    small_shift = 1  # metre
    slightly_shifted_projected_scope = projected_scope.offset_curve(small_shift)

    # a small shift by a positive value should move the line closer to the river centerline
    distance_multiplier = 1 if river_centerline.distance(slightly_shifted_projected_scope) < river_centerline.distance(projected_scope) else -1

    return projected_scope.offset_curve(distance_multiplier * distance)


def process_prediction(prediction_dataframe, measurement_year: int = 2024) -> gpd.GeoDataFrame:
    """Turn the prediction into a geodataframe with timestamps."""
    TIME_COLUMN = "time_column"
    DISTANCE = "distance_to_erosion"
    YEAR = "year"
    TIMESTAMP = "timestamp"
    
    processed_prediction = pd.melt(prediction_dataframe, var_name=TIME_COLUMN, value_name=DISTANCE, ignore_index=False).reset_index().copy()
    processed_prediction[YEAR] = processed_prediction[TIME_COLUMN].map(lambda x: int(x.split('_')[-1]) + measurement_year)
    processed_prediction[TIMESTAMP] = processed_prediction[YEAR].map(lambda x: f"{x}-1-1")
    processed_prediction[TIMESTAMP] = pd.to_datetime(processed_prediction[TIMESTAMP])
    processed_prediction.drop([CONST.TIMESTAMP, TIME_COLUMN, YEAR], axis=1, inplace=True)

    return processed_prediction

print("✅ Helper functions defined!")

✅ Helper functions defined!


In [81]:
# Process predictions into geodataframe with projected river banks
print("🗺️  Processing predictions for visualization...")

processed_prediction = process_prediction(predictions)
processed_prediction = processed_prediction.merge(prediction_regions, on=CONST.PREDICTION_REGION_ID)
processed_prediction.rename(columns={"geometry": "prediction_region"}, inplace=True)

# Load river centerline for calculating projected river bank positions
# NOTE: Use the erosion-specific centerline from Levering_erosie_data.gpkg, not the general middenlijn
etienne_geospatial_data = PATHS.DATA_DIR / "Levering_erosie_data.gpkg"
centerline_layer_name = "Centreline_River"
centerline = gpd.read_file(etienne_geospatial_data, layer=centerline_layer_name)
centerline.to_crs(prediction_regions.crs, inplace=True)

# Get the relevant (closest) centerline segment for each prediction region
processed_prediction["relevant_centerline"] = processed_prediction["prediction_region"].map(
    lambda x: U.get_relevant_centerline(x, centerline)
)

# Project each prediction region onto the erosion border
processed_prediction["projected_scope"] = processed_prediction["prediction_region"].map(
    lambda x: project_scope_onto_erosion_border(x, erosion_border)
)

# Calculate where the predicted river bank would be based on the distance
processed_prediction["predicted_riverbank"] = processed_prediction.apply(
    lambda row: move_projected_scope_to_side(
        row["projected_scope"], 
        row["distance_to_erosion"], 
        row["relevant_centerline"]
    ),
    axis=1
)

# Clean up intermediate columns and create final GeoDataFrame
processed_prediction.drop(["prediction_region", "relevant_centerline", "projected_scope"], axis=1, inplace=True)
processed_prediction_gdf = gpd.GeoDataFrame(processed_prediction, geometry="predicted_riverbank", crs=prediction_regions.crs)

print(f"✅ Processed {len(processed_prediction)} prediction records")
processed_prediction_gdf.head()

🗺️  Processing predictions for visualization...
✅ Processed 5390 prediction records


,location_id,distance_to_erosion,timestamp,start_year,end_year,predicted_riverbank
0,ijssel_l_9509_9510,90575.142575,2025-01-01,2017,2022,LINESTRING EMPTY
1,ijssel_l_9509_9510,90573.884011,2025-01-01,2017,2022,LINESTRING EMPTY
2,ijssel_l_9510_9511,90581.965323,2025-01-01,2017,2022,LINESTRING EMPTY
3,ijssel_l_9510_9511,90581.568781,2025-01-01,2017,2022,LINESTRING EMPTY
4,ijssel_l_9520_9521,90902.740899,2025-01-01,2017,2022,LINESTRING EMPTY


## Interactive Map with Time-based Animation

This map shows:
- **Prediction regions** (blue = safe, orange = crosses erosion border)
- **Predicted river bank** positions over time (animated)
- **Erosion border** (critical limit line)
- **Interactive charts** (click on regions to see erosion trends)

In [80]:
# Create interactive folium map
print("🗺️  Creating interactive map...")

mapa = folium.Map(
    location=[CONST.CENTRE_NL_LAT, CONST.CENTRE_NL_LON], 
    zoom_start=CONST.DEFAULT_NL_ZOOM, 
    control_scale=True
)

# Layer 1: Predicted river bank positions (time-animated)
fg_bank = folium.FeatureGroup(name="predicted river bank", show=False).add_to(mapa)

geojson_features = []
for _, row in processed_prediction_gdf.to_crs(epsg=CONST.EPSG_WGS84).iterrows():
    line = row["predicted_riverbank"].interpolate(0.5, normalized=True).__geo_interface__
    geojson_features.append({
        "type": "Feature",
        "geometry": line,
        "properties": {
            "times": [row["timestamp"].strftime("%Y-%m-%dT%H:%M:%S")],
            "style": {
                "color": "green" if row["distance_to_erosion"] > 0 else "red", 
                "weight": 3, 
                "opacity": 0.6
            },
        },
    })

geojson_data = {"type": "FeatureCollection", "features": geojson_features}
TimestampedGeoJson(
    geojson_data,
    period="P1Y",
    duration="P6M",
    transition_time=200,  # Milliseconds between frames
    loop=False,
    auto_play=False,
    loop_button=True,
).add_to(mapa)

# Layer 2: Erosion border (critical limit line)
fg_border = folium.FeatureGroup(name="erosion border", show=False).add_to(mapa)
folium.GeoJson(erosion_border_gdf["geometry"].to_crs(epsg=CONST.EPSG_WGS84)).add_to(fg_border)

# Layer 3: Prediction regions with interactive charts
fg_scope = folium.FeatureGroup(name="prediction regions", show=True).add_to(mapa)

measurement_year = 2024
predicted_years = [measurement_year + int(col.split("_")[-1]) for col in predictions.columns]

regions_with_predictions = 0
regions_skipped = 0

for ind, row in prediction_regions.to_crs(CONST.EPSG_WGS84).iterrows():
    # Filter predictions for this region
    region_predictions = predictions[predictions.index.get_level_values(CONST.PREDICTION_REGION_ID) == row[CONST.PREDICTION_REGION_ID]]
    
    # Skip regions without erosion data/predictions
    if len(region_predictions) == 0:
        regions_skipped += 1
        continue
    
    regions_with_predictions += 1
    predicted_values = region_predictions.iloc[0].values
    data = pd.DataFrame({"year": predicted_years, "distance to erosion border (m)": predicted_values})
    color = "orange" if (data["distance to erosion border (m)"] < 0).any() else "blue"

    # Create altair chart
    chart = alt.Chart(data, title=row[CONST.PREDICTION_REGION_ID]).mark_line(point=True).encode(
        x="year", 
        y="distance to erosion border (m)"
    )
    horizontal_line = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[5, 5], color="black").encode(y="y")
    chart = chart + horizontal_line
    
    vega_chart = folium.VegaLite(chart, width="100%", height="100%")
    popup = folium.Popup()
    vega_chart.add_to(popup)

    polygon = folium.GeoJson(
        row["geometry"],
        style_function=lambda feature, color=color: {"color": color, "fillcolor": color}
    )
    popup.add_to(polygon)
    polygon.add_to(fg_scope)

folium.LayerControl().add_to(mapa)

print(f"✅ Map created!")
print(f"   - Regions with predictions: {regions_with_predictions}")
print(f"   - Regions without erosion data: {regions_skipped}")
print(f"\n💡 Click on regions to see erosion trends over time")
print(f"💡 Use layer control (top right) to toggle visibility")

mapa

🗺️  Creating interactive map...
✅ Map created!
   - Regions with predictions: 199
   - Regions without erosion data: 34

💡 Click on regions to see erosion trends over time
💡 Use layer control (top right) to toggle visibility


---

# Complete Pipeline Summary

In [74]:
print("="*80)
print("🎉 COMPLETE END-TO-END PIPELINE FINISHED!")
print("="*80)
print()
print(f"📂 Input:  {input_path.name}")
print(f"📂 Output: {output_path.name}")
print()
print("✅ STEP 1: Data Collection (WFSDataBundler)")
print(f"   - Regions processed: {len(scope_regions)}")
print(f"   - WFS layers collected: {len(config.known_wfs_services)}")
print(f"   - Collection time: ~6.3 minutes (~1.6s per region)")
print()
print("✅ STEP 2: Feature Engineering (DataHandler)")
print(f"   - WFS features generated: {data_handler.scope_region_features.shape[1] - 4} columns")
print(f"   - Feature generation time: <1 second (~200x faster than live WFS)")
print()
print("✅ STEP 3: Erosion Analysis & Modeling")
print(f"   - River bank measurements: {len(river_bank_locations):,}")
print(f"   - Erosion features: {data_handler_full.processed_erosion_data.shape}")
print(f"   - Model: Baseline Erosion Model (trained)")
print(f"   - Predictions: 10 time steps ahead for {len(predictions)} regions")
print(f"   - Critical regions: {critical_regions}/{len(future_10)} ({100*critical_regions/len(future_10):.1f}%)")
print()
print("✅ STEP 4: Visualization")
print(f"   - Interactive map: {regions_with_predictions} regions with charts")
print(f"   - Animated erosion predictions over time")
print(f"   - Regions without erosion data: {regions_skipped}")
print()
print("🎯 Pipeline complete! Scroll up to see the interactive map.")
print("="*80)

🎉 COMPLETE END-TO-END PIPELINE FINISHED!

📂 Input:  phase1_2025-08-14_v1.gpkg
📂 Output: phase1_2025-08-14_v1_complete_20260127.gpkg

✅ STEP 1: Data Collection (WFSDataBundler)
   - Regions processed: 233
   - WFS layers collected: 3
   - Collection time: ~6.3 minutes (~1.6s per region)

✅ STEP 2: Feature Engineering (DataHandler)
   - WFS features generated: 7 columns
   - Feature generation time: <1 second (~200x faster than live WFS)

✅ STEP 3: Erosion Analysis & Modeling
   - River bank measurements: 70,495
   - Erosion features: (539, 2)
   - Model: Baseline Erosion Model (trained)
   - Predictions: 10 time steps ahead for 539 regions
   - Critical regions: 370/539 (68.6%)

✅ STEP 4: Visualization
   - Interactive map: 199 regions with charts
   - Animated erosion predictions over time
   - Regions without erosion data: 34

🎯 Pipeline complete! Scroll up to see the interactive map.


In [75]:
# Preview features
print("\n📊 First 5 Regions with Features:")
print("=" * 60)

data_handler.scope_region_features.head()


📊 First 5 Regions with Features:


,location_id,start_year,end_year,geometry,BrpGewas_area_fraction,BrpGewas_majority_class_category,BrpGewas_majority_class_gewas,bag:pand_area_fraction,bag:pand_majority_class_gebruiksdoel,rws_vegetatielegger:bomen_numerical_density,rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse
0,maas_l_2180_2181,2016,2024,"POLYGON ((148604.881 416309.328, 148538.207 41...",0.156551,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,NaN,0.000058,Bos
1,maas_l_2181_2182,2016,2024,"POLYGON ((148491.678 416329.661, 148416.977 41...",0.370831,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,NaN,0.000056,Water
2,maas_l_2182_2183,2016,2024,"POLYGON ((148373.399 416364.687, 148290.662 41...",0.146087,Grasland,"Grasland, blijvend",0.0,NaN,0.000000,Gras en Akker
3,maas_l_2183_2184,2016,2024,"POLYGON ((148268.941 416410.637, 148182.141 41...",0.327541,Grasland,"Grasland, blijvend",0.0,NaN,0.000063,Riet en Ruigte
4,maas_l_2184_2185,2016,2024,"POLYGON ((148182.141 416457.702, 148095.34 416...",0.254791,Grasland,"Grasland, blijvend",0.0,NaN,0.000127,Riet en Ruigte


---

# Export Results

In [27]:
# Export features to CSV (optional)
csv_path = output_path.with_suffix('.csv')

# Drop geometry for CSV export
features_df = data_handler.scope_region_features.drop(columns=['geometry'])
features_df.to_csv(csv_path, index=False)

print(f"💾 Features exported to: {csv_path}")
print(f"📊 Shape: {features_df.shape}")

💾 Features exported to: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/phase1_2025-08-14_v1_complete_20260127.csv
📊 Shape: (233, 10)


---

# Pipeline Summary

In [28]:
print("="*60)
print("PIPELINE COMPLETE!")
print("="*60)
print()
print(f"📂 Input:  {input_path.name}")
print(f"📂 Output: {output_path.name}")
print(f"📂 CSV:    {csv_path.name}")
print()
print(f"📊 Regions processed: {len(scope_regions)}")
print(f"📊 Features generated: {len(feature_cols)}")
print(f"📊 Output shape: {data_handler.scope_region_features.shape}")
print()
print("✅ Data ready for machine learning!")

PIPELINE COMPLETE!

📂 Input:  phase1_2025-08-14_v1.gpkg
📂 Output: phase1_2025-08-14_v1_complete_20260127.gpkg
📂 CSV:    phase1_2025-08-14_v1_complete_20260127.csv

📊 Regions processed: 233
📊 Features generated: 7
📊 Output shape: (233, 11)

✅ Data ready for machine learning!
